LTSM on MNIST

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import struct
import numpy as np

LTSM model

In [2]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=28, hidden_size=256, layer_dim=1, classes_count=10):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_size, hidden_size, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_size, classes_count)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_size).to(x.device)
        out, states = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

Train and test functions

In [3]:
def train_model(model, train_loader, criterion, optimizer, epochs, device):
    for epoch in range(epochs):
        model.train()
        current_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.reshape(-1, 28, 28).to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            current_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = current_loss / total
        epoch_acc = 100.0 * correct / total

        print(f'Epoch {epoch+1} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%')

def evaluate_model(model, test_loader, criterion, device):
    model.eval()
    current_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.reshape(-1, 28, 28).to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            current_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_loss = current_loss / total
    test_acc = 100.0 * correct / total

    print(f'Test Loss: {test_loss:.4f} | Test Accuracy: %{test_acc:.2f}')

Load dataset

In [4]:
def load_idx_images(path):
    with open(path, 'rb') as f:
        _, n, rows, cols = struct.unpack('>IIII', f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, 1, rows, cols).astype(np.float32) / 255.0


def load_idx_labels(path):
    with open(path, 'rb') as f:
        _, n = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n)


def get_train_loader(batch_size=128):
    images = load_idx_images('MNIST/train-images.idx3-ubyte')
    labels = load_idx_labels('MNIST/train-labels.idx1-ubyte')

    images = torch.from_numpy(images)
    labels = torch.from_numpy(labels).long()

    dataset = TensorDataset(images, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


def get_test_loader(batch_size=128):
    images = load_idx_images('MNIST/t10k-images.idx3-ubyte')
    labels = load_idx_labels('MNIST/t10k-labels.idx1-ubyte')

    images = torch.from_numpy(images)
    labels = torch.from_numpy(labels).long()

    dataset = TensorDataset(images, labels)
    return DataLoader(dataset, batch_size=batch_size)


train_loader = get_train_loader(batch_size=128)
test_loader = get_test_loader(batch_size=128)
print(f'Train: {len(train_loader.dataset)} images')
print(f'Test:  {len(test_loader.dataset)} images')


Train: 60000 images
Test:  10000 images


/tmp/ipykernel_319269/3915163411.py:20: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  labels = torch.from_numpy(labels).long()


Train and Test model

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Hidden layer size: 256")
model = LSTMClassifier(hidden_size=256).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
train_model(model, train_loader, criterion, optimizer, epochs=10, device=device)
evaluate_model(model, test_loader, criterion, device)

Hidden layer size: 256
Epoch 1 | Loss: 0.4561 | Accuracy: 85.09%
Epoch 2 | Loss: 0.1055 | Accuracy: 97.07%
Epoch 3 | Loss: 0.0752 | Accuracy: 97.88%
Epoch 4 | Loss: 0.0625 | Accuracy: 98.23%
Epoch 5 | Loss: 0.0532 | Accuracy: 98.46%
Epoch 6 | Loss: 0.0510 | Accuracy: 98.53%
Epoch 7 | Loss: 0.0448 | Accuracy: 98.67%
Epoch 8 | Loss: 0.0456 | Accuracy: 98.64%
Epoch 9 | Loss: 0.0434 | Accuracy: 98.72%
Epoch 10 | Loss: 0.0456 | Accuracy: 98.63%
Test Loss: 0.0700 | Test Accuracy: %98.02
